In [6]:
import os
import subprocess
from tqdm import tqdm
import math

# --- Configuration ---
# Base directory containing your 1st_10min, 2nd_10min, etc., folders
INPUT_BASE_DIR = "/home/poorna/Downloads/dataa"  # <-- **UPDATE THIS PATH**
OUTPUT_BASE_DIR = "/home/poorna/Downloads/data_frame" # <-- **UPDATE THIS PATH**

# List of the 7 source directories
SOURCE_DIRS = [f"{i}st_10min" if i == 1 else f"{i}th_10min" for i in range(1, 8)]
SOURCE_DIRS[1] = "2nd_10min" # Corrects 2nd
SOURCE_DIRS[2] = "3rd_10min" # Corrects 3rd

# --- Helper Function: Get Video Duration (Requires ffprobe) ---

def get_video_duration(filepath):
    """Uses ffprobe to get the video duration in seconds."""
    try:
        # Command to probe the file for duration
        command = [
            'ffprobe',
            '-v', 'error',
            '-show_entries', 'format=duration',
            '-of', 'default=noprint_wrappers=1:nokey=1',
            filepath
        ]
        
        # Execute the command and capture output
        result = subprocess.run(
            command, 
            capture_output=True, 
            text=True, 
            check=True
        )
        
        # Parse the duration (which is a float string)
        duration = float(result.stdout.strip())
        return duration
        
    except FileNotFoundError:
        print("\nERROR: 'ffprobe' not found. Ensure FFmpeg is installed and in your system PATH.")
        raise
    except Exception as e:
        # Handle case where ffprobe fails to read a video property
        # print(f"Warning: Could not get duration for {filepath}: {e}")
        return None

# --- Processing Function ---

def extract_middle_frames_ffmpeg(input_base, output_base, source_dirs):
    """
    Extracts the middle frame from every video clip using FFmpeg's -ss (seek) command.
    """
    if not os.path.exists(output_base):
        os.makedirs(output_base)
        print(f"Created output base directory: {output_base}")

    for source_dir_name in source_dirs:
        input_path = os.path.join(input_base, source_dir_name)
        
        output_dir_name = f"{source_dir_name}_mid"
        output_path = os.path.join(output_base, output_dir_name)

        if not os.path.exists(output_path):
            os.makedirs(output_path)
            print(f"\nCreating output folder: {output_dir_name}")

        # Find all video files
        clip_files = [f for f in os.listdir(input_path) if f.endswith(('.mp4', '.avi', '.mov'))]

        if not clip_files:
            print(f"Warning: No video files found in {input_path}. Skipping.")
            continue

        for clip_filename in tqdm(clip_files, desc=f"Processing {source_dir_name}"):
            clip_filepath = os.path.join(input_path, clip_filename)
            
            # 1. Get duration
            duration = get_video_duration(clip_filepath)

            if duration is None or duration <= 0:
                continue

            # 2. Calculate middle time point
            middle_time = duration / 2.0
            
            # Format middle time to 3 decimal places for precise seeking
            middle_time_str = f"{middle_time:.3f}"

            # 3. Define output path
            base_name = os.path.splitext(clip_filename)[0]
            output_filename = f"{base_name}_mid.jpg"
            output_filepath = os.path.join(output_path, output_filename)
            
            # FFmpeg Command Structure:
            # -ss {time} : Seeks to the middle time point (precise)
            # -i {input} : Specifies the input file
            # -vframes 1 : Extracts exactly one frame
            # -q:v 2    : Sets video quality (1=best, 31=worst)
            # {output}  : Specifies the output JPEG file
            ffmpeg_command = [
                'ffmpeg',
                '-ss', middle_time_str,  # Seek to middle time
                '-i', clip_filepath,     # Input file
                '-vframes', '1',
                '-q:v', '2',
                output_filepath
            ]

            # 4. Execute FFmpeg command
            try:
                subprocess.run(ffmpeg_command, check=True, capture_output=True, text=True)
            except subprocess.CalledProcessError as e:
                print(f"\nError processing {clip_filename}. FFmpeg output:")
                print(e.stderr)

    print("\n--- Extraction Complete! ---")

# --- Execution ---
if __name__ == "__main__":
    # NOTE: You MUST have FFmpeg (which includes ffprobe) installed and accessible
    # in your system's PATH for this script to work.
    extract_middle_frames_ffmpeg(INPUT_BASE_DIR, OUTPUT_BASE_DIR, SOURCE_DIRS)

Processing 2nd_10min: 100%|██████████| 200/200 [00:52<00:00,  3.83it/s]



Creating output folder: 3rd_10min_mid


Processing 3rd_10min: 100%|██████████| 200/200 [01:59<00:00,  1.68it/s]



Creating output folder: 4th_10min_mid


Processing 4th_10min: 100%|██████████| 200/200 [01:57<00:00,  1.70it/s]


...


Creating output folder: 6th_10min_mid


Processing 6th_10min: 100%|██████████| 200/200 [01:57<00:00,  1.70it/s]



Creating output folder: 7th_10min_mid


Processing 7th_10min: 100%|██████████| 200/200 [01:58<00:00,  1.69it/s]


--- Extraction Complete! ---


In [ ]:
INPUT_BASE_DIR = "home/poorna/Downloads/dataa/1st_10min"  # <-- **UPDATE THIS PATH**
OUTPUT_BASE_DIR = "home/poorna/Downloads/data_frame" # <-- **UPDATE THIS PATH**
